# Business purpose — public renewable assets and site geography

This notebook helps an energy portfolio operator understand the scale, technology mix and geographic concentration of operational public REPD projects before selecting assets for renewable forecasting research. REPD records are **public metadata**, not simulated customer sites.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from gridmatch.research.assets import asset_summaries, save_asset_artifacts
from gridmatch.research.common import project_path

np.random.seed(20260725)
repd = pd.read_parquet(project_path('data', 'processed', 'repd_operational_sites.parquet'))
assert set(repd['data_origin']) == {'public'}
summaries = asset_summaries(repd)
artifact_paths = save_asset_artifacts(repd)
display(summaries['technology'].head(15))
display(summaries['map_ready'])
print({name: str(path) for name, path in artifact_paths.items()})

,technology,project_count,installed_capacity_mw,median_project_mw
21,Wind Onshore,778,15271.45,9.20
20,Wind Offshore,48,15129.00,196.80
18,Solar Photovoltaics,1393,10918.06,5.00
2,Battery,171,4754.55,23.40
4,Biomass (dedicated),81,3466.19,9.00
14,Pumped Storage Hydroelectricity,4,2828.00,400.00
6,EfW Incineration,60,1480.20,19.30
11,Landfill Gas,270,754.70,2.00
3,Biomass (co-firing),2,663.00,331.50
12,Large Hydro,25,520.60,15.00


,total_operational_projects,valid_coordinates,invalid_coordinates,valid_coordinate_rate
0,3100,3096,4,0.99871


{'technology_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\01_repd_technology_summary.csv', 'capacity_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\01_repd_capacity_summary.csv', 'region_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\01_repd_region_summary.csv', 'map_ready_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\01_repd_map_ready_summary.csv', 'technology_figure': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\figures\\01_repd_capacity_by_technology.png', 'geography_figure': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\figures\\01_repd_operational_geography.png'}


## Capacity, project size and regional concentration

Counts alone overstate technologies with many small projects, so the next tables separate project count, total MW and size quantiles. Region is retained from the official source; coordinates are converted from British National Grid by the Phase 3 pipeline rather than inside this notebook.

In [2]:
display(summaries['capacity'].head(15))
display(summaries['region'].head(12))
solar_wind = summaries['technology'][summaries['technology']['technology'].astype(str).str.contains('solar|wind', case=False, regex=True)]
display(solar_wind)

,technology,p25_mw,p50_mw,p75_mw,p90_mw
0,Advanced Conversion Technologies,6.325,10.00,13.625,21.85
1,Anaerobic Digestion,1.200,1.85,2.400,4.05
2,Battery,7.500,23.40,49.900,50.00
3,Biomass (co-firing),174.750,331.50,488.250,582.30
4,Biomass (dedicated),2.600,9.00,25.250,43.12
5,Compressed Air Energy Storage,NaN,NaN,NaN,NaN
6,EfW Incineration,10.950,19.30,31.000,51.80
7,Flywheels,400.000,400.00,400.000,400.00
8,Geothermal,NaN,NaN,NaN,NaN
9,Hot Dry Rocks (HDR),3.000,3.00,3.000,3.00


,region,project_count,installed_capacity_mw
6,Offshore,54,15163.50
7,Scotland,567,12718.93
12,Yorkshire and Humber,170,4631.32
10,Wales,244,4569.00
9,South West,536,4172.69
8,South East,321,3776.25
1,Eastern,335,3628.00
0,East Midlands,272,2146.96
4,North West,196,1622.56
5,Northern Ireland,118,1453.50


,technology,project_count,installed_capacity_mw,median_project_mw
21,Wind Onshore,778,15271.45,9.2
20,Wind Offshore,48,15129.00,196.8
18,Solar Photovoltaics,1393,10918.06,5.0


## Public-data boundary

REPD shows planning/development metadata and operational status. It does **not** imply that every project exposes public half-hourly generation, a usable BM Unit mapping, availability, curtailment or commercial terms. The separate 12-site portfolio used elsewhere is explicitly **simulated**.

## Findings, limitations and production implications

**Findings:** technology, capacity, size, region and coordinate-validity summaries are reproducible from the cached public artifact; solar and wind form clear forecast-relevant subsets. **Limitations:** operational status and coordinates can be stale or incomplete, and REPD does not guarantee meter history. **Production implications:** version each extract, retain invalid-coordinate flags, validate prospective assets independently and do not infer output history from planning metadata.